In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("loan_approval_data.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'loan_approval_data.csv'

In [ ]:
df.info()
df.isnull().sum()
df.describe()

# Handle Missing Value

In [ ]:
categorical_cols = df.select_dtypes(include = ["object"]).columns
numerical_cols = df.select_dtypes(include = ["number"]).columns

In [ ]:
# To Eassily Handle missing value
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

In [ ]:
df.head()
df.isnull().sum()

In [ ]:
cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

# Exploratory Data Analysis

In [ ]:
# How balance our classes are?
classes_count = df["Loan_Approved"].value_counts()
plt.pie(classes_count, labels=["No","Yes"], autopct = "%1.1f%%")
plt.title("Is Loan Approved Or Not")

In [ ]:
# Analyze Categgorical Data
# gender_cnt = df["Gender"].value_counts()
# ax = sns.barplot(gender_cnt)
# ax.bar_label(ax.containers[0])

# edu_cnt = df["Education_Level"].value_counts()
# ax = sns.barplot(edu_cnt)
# ax.bar_label(ax.containers[0])


emp_cnt = df["Employer_Category"].value_counts()
ax = sns.barplot(emp_cnt)
ax.bar_label(ax.containers[0])

In [ ]:
# Analyze Income
sns.histplot(
    data = df,
    x="Applicant_Income",
    bins=20
)

In [ ]:
sns.histplot(
    data = df,
    x="Coapplicant_Income",
    bins=20
)

In [ ]:
# Outliers - Box Plot 
sns.boxplot(
    data = df,
    x= "Loan_Approved",
    y= "Applicant_Income"

)

In [ ]:
fig,axes = plt.subplots(3, 2, figsize=(12, 12))
sns.boxplot( ax = axes[0, 0],data = df, x= "Loan_Approved", y= "Applicant_Income")
sns.boxplot( ax = axes[0, 1],data = df, x= "Loan_Approved", y= "Credit_Score")
sns.boxplot( ax = axes[1, 0],data = df, x= "Loan_Approved", y= "DTI_Ratio")
sns.boxplot( ax = axes[1, 1],data = df, x= "Loan_Approved", y= "Savings")
sns.boxplot( ax = axes[2, 0],data = df, x= "Loan_Approved", y= "Age")
sns.boxplot( ax = axes[2, 1],data = df, x= "Loan_Approved", y= "Loan_Amount")
fig.tight_layout()

In [ ]:
# Credit Score with Loan Approved
sns.histplot(
    data = df,
    x="Credit_Score",
    bins=20,
    hue ="Loan_Approved",
    multiple = "dodge"
)

In [ ]:
sns.histplot(
    data = df,
    x="Applicant_Income",
    bins=20,
    hue ="Loan_Approved",
    multiple = "dodge"
)

In [ ]:
#Remove Applicant Id
df = df.drop("Applicant_ID", axis=1)

In [ ]:
df.head()

# Features Encoding

In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])


In [ ]:
df.head()

In [ ]:
cols = ["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")

encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns = ohe.get_feature_names_out(cols), index = df.index)

df = pd.concat([df.drop(columns=cols), encoded_df], axis = 1)

In [ ]:
df.head()
df.info()

# Correlation Heatmap

In [ ]:
num_cols = df.select_dtypes(include="number")
corr_matrix = num_cols.corr()

plt.figure(figsize=(15,8))
sns.heatmap(
     corr_matrix,
     annot = True,
     fmt =".2f",
     cmap = "coolwarm"
 )

In [ ]:
# corr_matrix["Loan_Approved"].sort_values(ascending=False)

# Train-Test-Split and Feature Scalling

In [ ]:
X = df.drop("Loan_Approved",axis = 1)
y = df["Loan_Approved"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

# Train and Evaluate Models

In [ ]:
# Logistic Regression 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
log_model = LogisticRegression()

log_model.fit(X_train_scaled,y_train)

y_pred =log_model.predict(X_test_scaled) 

# Evaluation
print("Logistic Regression")
print("Accuracy: ",accuracy_score(y_test, y_pred))
print("Precision: ",precision_score(y_test, y_pred))
print("recall: ",recall_score(y_test, y_pred))
print("F1: ",f1_score(y_test, y_pred))
print("Cm: ",confusion_matrix(y_test, y_pred))

In [ ]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
Knn_model = KNeighborsClassifier(n_neighbors=5)

Knn_model.fit(X_train_scaled,y_train)

y_pred =Knn_model.predict(X_test_scaled) 

# Evaluation
print("KNN Model")
print("Accuracy: ",accuracy_score(y_test, y_pred))
print("Precision: ",precision_score(y_test, y_pred))
print("recall: ",recall_score(y_test, y_pred))
print("F1: ",f1_score(y_test, y_pred))
print("Cm: ",confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()

nb_model.fit(X_train_scaled,y_train)

y_pred =nb_model.predict(X_test_scaled) 

# Evaluation
print("Naive Bayes Model")
print("Accuracy: ",accuracy_score(y_test, y_pred))
print("Precision: ",precision_score(y_test, y_pred))
print("recall: ",recall_score(y_test, y_pred))
print("F1: ",f1_score(y_test, y_pred))
print("Cm: ",confusion_matrix(y_test, y_pred))

# Best Model on the basis of Precision => Naive Bayes

# Feature Engineering

In [ ]:
# Add and Transform Features
df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

# Is's do when you have Skew feature
# df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"]) 

X = df.drop(columns=["Loan_Approved","DTI_Ratio","Credit_Score"])
y = df["Loan_Approved"]

# train_test_split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

# Scalling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
# Logistic Regression 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
log_model = LogisticRegression()

log_model.fit(X_train_scaled,y_train)

y_pred =log_model.predict(X_test_scaled) 

# Evaluation
print("Logistic Regression")
print("Accuracy: ",accuracy_score(y_test, y_pred))
print("Precision: ",precision_score(y_test, y_pred))
print("recall: ",recall_score(y_test, y_pred))
print("F1: ",f1_score(y_test, y_pred))
print("Cm: ",confusion_matrix(y_test, y_pred))

In [ ]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
Knn_model = KNeighborsClassifier(n_neighbors=5)

Knn_model.fit(X_train_scaled,y_train)

y_pred =Knn_model.predict(X_test_scaled) 

# Evaluation
print("KNN Model")
print("Accuracy: ",accuracy_score(y_test, y_pred))
print("Precision: ",precision_score(y_test, y_pred))
print("recall: ",recall_score(y_test, y_pred))
print("F1: ",f1_score(y_test, y_pred))
print("Cm: ",confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()

nb_model.fit(X_train_scaled,y_train)

y_pred =nb_model.predict(X_test_scaled) 

# Evaluation
print("Naive Bayes Model")
print("Accuracy: ",accuracy_score(y_test, y_pred))
print("Precision: ",precision_score(y_test, y_pred))
print("recall: ",recall_score(y_test, y_pred))
print("F1: ",f1_score(y_test, y_pred))
print("Cm: ",confusion_matrix(y_test, y_pred))